## Evaluating the Homer TF-peak scoring method

Homer `annotatePeaks.pl` attempts to find TF motif binding sites in a provided set of peak locations. The output of `annotatePeaks.pl` is a file for each TF with information about its binding potential to the peaks in your dataset. The columns are:

- PeakID
- Chromosome
- Peak start position
- Peak end position
- Strand
- Peak Score
- FDR/Peak Focus Ratio/Region Size
- Annotation (i.e. Exon, Intron, ...)
- Detailed Annotation (Exon, Intron etc. + CpG Islands, repeats, etc.)
- Distance to nearest RefSeq TSS
- Nearest TSS: Native ID of annotation file
- Nearest TSS: Entrez Gene ID
- Nearest TSS: Unigene ID
- Nearest TSS: RefSeq ID
- Nearest TSS: Ensembl ID
- Nearest TSS: Gene Symbol
- Nearest TSS: Gene Aliases
- Nearest TSS: Gene description
- Additional columns depend on options selected when running the program.

To test the accuracy of Homer, we can have it predict the TF binding sites for ChIP-seq TF-DNA binding interactions. We can compare the TF-DNA Homer predictions to the known binding interactions to determine how well Homer identifies correct TF-DNA binding partners.

So far, we have been evaluating our results against the mESC ground truth from the BEELINE paper. The paper does not provide the TF-peak ChIP-seq file, but rather the direct TF-TG mapping. The paper states that the ChIP-seq dataset was downloaded from ChIP-Atlas, so we are using the **mm10 embryo ChIP: TFs and others** dataset from ChIP-Atlas.

In [1]:
!hostnamectl

   Static hostname: psh01com1hcom40
         Icon name: computer-server
           Chassis: server
        Machine ID: 4669d2030ba24ac48f90d1c4189391ec
           Boot ID: 1ffc2fe7310b49648c02326d4a20a716
  Operating System: ]8;;https://www.redhat.com/Red Hat Enterprise Linux 8.10 (Ootpa)]8;;
       CPE OS Name: cpe:/o:redhat:enterprise_linux:8::baseos
            Kernel: Linux 4.18.0-553.115.1.el8_10.x86_64
      Architecture: x86-64


In [1]:
import os
import pandas as pd
import pybedtools
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/")
cell_type = "mESC"
sample_name = "E7.5_rep1"
experiment_name = f"{cell_type}_{sample_name}_tutorial"

atac_file = DATA_DIR / f"PROCESSED_DATA/{experiment_name}/{sample_name}/RE_pseudobulk.parquet"
atac_df = pd.read_parquet(atac_file)
atac_peaks = atac_df.index.to_series()


In [2]:
def format_peaks(peak_ids: pd.Series) -> pd.DataFrame:
    """
    Splits peaks from `chrN:start-end` format into a DataFrame.
    
    Creates a dataframe with the following columns:
    1) "peak_id": peakN+1 where N is the index position of the peak
    2) "chromosome": chrN
    3) "start"
    4) "end"
    5) "strand": List of "." values, we dont have strand information for our peaks.
    
    Args:
        peak_ids (pd.Series):
            Series containing the peak locations in "chrN:start-end" format.
            
    Returns:
        peak_df (pd.DataFrame):
            DataFrame of peak locations in the correct format for Homer and the sliding window method
    """
    if peak_ids.empty:
        raise ValueError("Input peak ID list is empty.")
    
    peak_ids = peak_ids.drop_duplicates()

    print(f'Formatting {peak_ids.shape[0]} peaks')

    # Extract chromosome, start, and end from peak ID strings
    try:
        chromosomes = peak_ids.str.extract(r'([^:]+):')[0]
        starts = peak_ids.str.extract(r':(\d+)-')[0]
        ends = peak_ids.str.extract(r'-(\d+)$')[0]
    except Exception as e:
        raise ValueError(f"Error parsing 'peak_id' values: {e}")

    if chromosomes.isnull().any() or starts.isnull().any() or ends.isnull().any():
        raise ValueError("Malformed peak IDs. Expect format 'chr:start-end'.")

    peak_df = pd.DataFrame({
        "peak_id": [f"peak{i + 1}" for i in range(len(peak_ids))],
        "chromosome": chromosomes,
        "start": pd.to_numeric(starts, errors='coerce').astype(int),
        "end": pd.to_numeric(ends, errors='coerce').astype(int),
        "strand": ["."] * len(peak_ids)
    })
    
    return peak_df

Next, we need to format the peaks to follow the requirements for the Homer peaks file.

> HOMER peak files should have at minimum 5 columns (separated by TABs, additional columns will be ignored:
> - Column1: Unique Peak ID
> - Column2: chromosome
> - Column3: starting position
> - Column4: ending position
> - Column5: Strand (+/- or 0/1, where 0="+", 1="-")

If we save the ChIP-seq peaks as `homer_peaks.txt` to the `tmp` directory of the output folder, then they will be used to calculate the sliding window and Homer scores

In [3]:
peak_df = format_peaks(atac_peaks)

tmp_dir = "/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/dev/Homer/tmp"
os.makedirs(tmp_dir, exist_ok=True)

homer_peak_path = os.path.join(tmp_dir, "homer_peaks.txt")
peak_df.to_csv(homer_peak_path, sep="\t", header=False, index=False)

Formatting 199885 peaks


### Running Homer

Now that we have created the ground truth TF-peak and `homer_peaks.txt` files, we can run Homer on the ChIP-seq peaks

In [5]:
!sbatch /gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/dev/Homer/run_homer_on_chipseq.sh

Submitted batch job 3656025
